# DSI490 — Flood Chat (Agent 1) · ทดสอบแบบ guy01

คุยกับ agent ทีละข้อความ เหมือนผู้ใช้พิมพ์ใน LINE
```python
output = agent(input_text="...")
print(output.output_text)
```
- `agent.show_last()` ดูว่า turn ล่าสุดเกิดอะไรขึ้น (Gemini ดึงอะไร, initial/correction, flag, เหตุผล)
- `agent.show_state()` ดูข้อมูลเหตุการณ์ที่เก็บไว้
- `agent.reset()` เริ่มบทสนทนาใหม่

**เรียก Gemini 1 ครั้งต่อข้อความ** (Extraction) ข้อความตอบมาจาก `templates_th.json` ทั้งหมด

ยังไม่มีในตอนนี้: ส่งรายงานให้ Agent 2 จริง (ดูจาก `output.next_actions`), ฐานข้อมูล, LINE, RAG, จับคู่ชื่อพื้นที่

In [1]:
import os
import dspy
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or os.getenv("API_KEY")
model = os.getenv("GEMINI_MODEL", "gemini/gemini-3.1-flash-lite")
lm = dspy.LM(model=model, api_key=api_key)
dspy.configure(lm=lm)

In [3]:
import sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dspy-agent").is_dir())
sys.path.insert(0, str(ROOT / "dspy-agent"))

from app.agent import FloodAgent
from app.schemas import Hotline, now_bkk

print("model =", model, "| มี key:", bool(api_key))

model = gemini-3.1-flash-lite | มี key: True


สายด่วนด้านล่างเป็น**ค่าชั่วคราว ยังไม่ได้ตรวจกับแหล่งทางการ** (งานค้างของทีม) ของจริงต้องมาจาก config พร้อม `source_url` และ `verified_at` จริง

In [4]:
HOTLINES = [
    Hotline(key="ddpm", name="สายด่วน ปภ. (ยังไม่ได้ตรวจ)", number="1784", source_url="TODO", verified_at=now_bkk()),
    Hotline(key="ems", name="การแพทย์ฉุกเฉิน (ยังไม่ได้ตรวจ)", number="1669", source_url="TODO", verified_at=now_bkk()),
]

In [5]:
agent = FloodAgent(hotlines=HOTLINES)

## 1. เหตุทั่วไป: ถามทีละข้อ แล้วขอยืนยันก่อนส่ง

In [6]:
output = agent(input_text="น้ำท่วมบ้านค่ะ")

In [7]:
output.output_text

'รับทราบข้อมูลที่แจ้งค่ะ\n\nเหตุเกิดที่หมู่บ้าน ตำบล หรืออำเภอใด กรุณาบอกชื่อพื้นที่ค่ะ'

In [8]:
output = agent(input_text="ตำบลทับยา อินทร์บุรีค่ะ แค่อยากแจ้งไว้ก่อน")
print(output.output_text)

รับทราบข้อมูลที่แจ้งค่ะ

ขอจำนวนผู้ได้รับผลกระทบ และหากมีเด็ก ผู้สูงอายุ ผู้พิการ หญิงตั้งครรภ์ หรือผู้ป่วยที่ต้องดูแลเป็นพิเศษ ขอประเภทและจำนวนเท่าที่ทราบค่ะ


In [9]:
agent.show_last()

👤 ตำบลทับยา อินทร์บุรีค่ะ แค่อยากแจ้งไว้ก่อน
   ✦ intents=['report_flood', 'confirm_report'] target=active_incident
     - location_area: known "ตำบลทับยา อำเภออินทร์บุรี"
     - needs: known ["info_only"]
   • location_area: initial
   • needs: initial
   ➜ actions=['ask_follow_up']
   ℹ️ ไม่รับ confirm_report: ยังไม่มีคำขอยืนยันที่ส่งสำเร็จและรอคำตอบ


In [10]:
output = agent(input_text="ไม่ทราบค่ะ")
print(output.output_text)

รับทราบว่ายังไม่ทราบข้อมูลส่วนนี้ค่ะ

หากทราบ ขอระดับน้ำที่เห็น จะบอกเป็นจุดเทียบกับร่างกายหรือความสูงที่วัดได้ก็ได้ค่ะ


In [11]:
output = agent(input_text="80 เซนติเมตร")
print(output.output_text)

รับทราบข้อมูลที่แจ้งค่ะ

สรุปข้อมูลที่แจ้งค่ะ
- พื้นที่: ตำบลทับยา อำเภออินทร์บุรี
- สถานการณ์: น้ำท่วมบ้านค่ะ
- ความต้องการ: แจ้งให้ทราบเท่านั้น
- ผู้ได้รับผลกระทบ: ไม่ทราบ
- ผู้ที่ต้องดูแลเป็นพิเศษ: ไม่ทราบ
- ระดับน้ำ: 80 เซนติเมตร
หากข้อมูลถูกต้อง กรุณายืนยันให้ส่งรายงาน หรือแจ้งส่วนที่ต้องการแก้ไขได้ค่ะ


In [12]:
output = agent(input_text="แก้เป็น 50 เซนติเมตรค่ะ พิมพ์ผิด")
print(output.output_text)
agent.show_last()

รับทราบการแก้ไขข้อมูลค่ะ

ข้อมูลมีการเปลี่ยนแปลงหลังสรุปครั้งก่อน กรุณาตรวจสรุปล่าสุดอีกครั้งค่ะ
สรุปข้อมูลที่แจ้งค่ะ
- พื้นที่: ตำบลทับยา อำเภออินทร์บุรี
- สถานการณ์: น้ำท่วมบ้านค่ะ
- ความต้องการ: แจ้งให้ทราบเท่านั้น
- ผู้ได้รับผลกระทบ: ไม่ทราบ
- ผู้ที่ต้องดูแลเป็นพิเศษ: ไม่ทราบ
- ระดับน้ำ: 50 เซนติเมตร
หากข้อมูลถูกต้อง กรุณายืนยันให้ส่งรายงาน หรือแจ้งส่วนที่ต้องการแก้ไขได้ค่ะ
👤 แก้เป็น 50 เซนติเมตรค่ะ พิมพ์ผิด
   ✦ intents=['correct_information'] target=active_incident
     - water_level: known {"raw": "50 เซนติเมตร", "descriptor": null, "cm": 50.0} (Gemini: แก้ไข)
   • water_level: correction
   ➜ actions=['confirm_report']
   ℹ️ ข้อมูลเปลี่ยนหลังแสดงสรุป ต้องยืนยันสรุปใหม่


In [13]:
output = agent(input_text="ถูกต้องค่ะ ส่งได้เลย")
print(output.output_text)
print(output.incident_status, output.next_actions)

ระบบรับรายงานนี้แล้ว และจะส่งต่อให้ผู้ดูแลรายงานพิจารณาค่ะ หากมีข้อมูลเปลี่ยนแปลง แจ้งเพิ่มเติมได้ค่ะ
IncidentStatus.SUBMITTED [<NextAction.SUBMIT_REPORT: 'submit_report'>]


In [14]:
agent.show_state()

incident inc_302a521b747a  status=submitted  matching=new  urgent=[]
  location_area          known       ถาม 1  {"raw": "ตำบลทับยา อำเภออินทร์บุรี", "stated_level": "unspecified", "match_status": "unverified", "candidates": []}
  location_landmark      not_asked   ถาม 0  
  location_coordinates   not_asked   ถาม 0  
  situation              known       ถาม 0  {"text": "น้ำท่วมบ้านค่ะ", "tags": ["water_in_house"]}
  needs                  known       ถาม 0  ["info_only"]
  affected_people        unknown     ถาม 1  
  vulnerable_people      unknown     ถาม 1  
  water_level            known       ถาม 1  {"raw": "50 เซนติเมตร", "cm": 50.0}  (เปลี่ยน 1 ครั้ง)
  contact                not_asked   ถาม 0  


## 2. เหตุด่วน: ส่งทันที แสดงสายด่วน แล้วถามต่อ (ไม่ขอยืนยัน)

เบอร์ในข้อความเป็นเบอร์สมมติ — ดูบรรทัด 🔒 ใน `show_last()` ว่า Gemini ไม่เห็นเบอร์

In [15]:
agent.reset()
output = agent(input_text="ช่วยด้วยค่ะ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก หมู่ 3 ตำบลน้ำตาล เบอร์ 081-234-5678")
print(output.output_text)
print(output.incident_status, output.next_actions)

จากข้อมูลที่แจ้ง กรุณาติดต่อช่องทางฉุกเฉินเพื่อขอความช่วยเหลือทันทีค่ะ
• สายด่วน ปภ. (ยังไม่ได้ตรวจ) 1784
• การแพทย์ฉุกเฉิน (ยังไม่ได้ตรวจ) 1669

รับทราบข้อมูลที่แจ้งค่ะ

ระบบรับรายงานนี้แล้ว และจะส่งต่อให้ผู้ดูแลรายงานพิจารณาทันทีค่ะ หากบอกข้อมูลเพิ่มเติมได้ จะช่วยให้รายงานชัดเจนขึ้นค่ะ

หากสะดวก ขอแชร์ตำแหน่งที่เกิดเหตุ เพื่อแนบพิกัดไปกับรายงานนี้ให้ผู้ดูแลรายงานใช้ประกอบการพิจารณา หากแจ้งแทนผู้อื่น กรุณาเลื่อนหมุดไปที่จุดเกิดเหตุแทนตำแหน่งของคุณ หากไม่สะดวก ข้ามหรือบอกจุดสังเกตแทนได้ค่ะ
IncidentStatus.SUBMITTED [<NextAction.SUBMIT_REPORT: 'submit_report'>, <NextAction.ASK_FOLLOW_UP: 'ask_follow_up'>]


In [16]:
agent.show_last()

👤 ช่วยด้วยค่ะ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก หมู่ 3 ตำบลน้ำตาล เบอร์ 081-234-5678
   🔒 ส่ง Gemini เป็น: ช่วยด้วยค่ะ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก หมู่ 3 ตำบลน้ำตาล เบอร์ [เบอร์โทร]
   ✦ intents=['report_flood', 'request_help'] target=new_incident
     - location_area: known "หมู่ 3 ตำบลน้ำตาล"
     - situation: known {"text": "น้ำขึ้นเร็วมาก", "tags": ["water_rising"]}
     - vulnerable_people: known {"present": true, "groups": [{"type": "elderly", "count": 1, "is_approximate": false}]}
     - contact: known "[เบอร์โทร]"
   • location_area: initial
   • situation: initial
   • vulnerable_people: initial
   keyword พบ: ['help_requested', 'people_trapped']
   ⚠️ flag ใหม่: ['help_requested', 'people_trapped']
   ➜ actions=['submit_report', 'ask_follow_up'] submit=urgent_flags
   ℹ️ พบเบอร์ 1 เบอร์: เก็บใน ContactRecord ไม่ส่ง Gemini


In [17]:
output = agent(input_text="อยู่หลังวัดค่ะ")
print(output.output_text)

รับทราบข้อมูลที่แจ้งค่ะ

มีผู้ได้รับผลกระทบหรือไม่ หากมี ขอจำนวนเท่าที่ทราบค่ะ


In [18]:
agent.show_state()

incident inc_bcaa39f2eaa2  status=submitted  matching=new  urgent=['help_requested', 'people_trapped']
  location_area          known       ถาม 0  {"raw": "หมู่ 3 ตำบลน้ำตาล", "stated_level": "unspecified", "match_status": "unverified", "candidates": []}
  location_landmark      known       ถาม 1  "หลังวัด"
  location_coordinates   unanswered  ถาม 1  
  situation              known       ถาม 0  {"text": "ช่วยด้วยค่ะ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก หมู่ 3 ตำบลน้ำตาล เบอร์ [เบอร์โทร]", "tags": ["water_rising"]}
  needs                  not_asked   ถาม 0  
  affected_people        not_asked   ถาม 1  
  vulnerable_people      known       ถาม 0  {"present": true, "groups": [{"type": "elderly", "count": 1, "is_approximate": false}]}
  water_level            not_asked   ถาม 0  
  contact                known       ถาม 0  true
  คำถามที่ค้าง: ['affected_people']


## 3. ลองเอง

พิมพ์ข้อความที่อยากทดสอบ แล้วรันซ้ำได้เรื่อย ๆ ใช้ `agent.reset()` เมื่ออยากเริ่มเรื่องใหม่

In [19]:
output = agent(input_text="ต้องตัดไฟตอนไหนคะ")
print(output.output_text)
agent.show_last()

ขณะนี้ยังไม่มีข้อมูลยืนยันเพียงพอสำหรับตอบคำถามนี้ ขอแนะนำสอบถามหน่วยงานที่เกี่ยวข้องผ่านช่องทางต่อไปนี้ค่ะ
• สายด่วน ปภ. (ยังไม่ได้ตรวจ) 1784
• การแพทย์ฉุกเฉิน (ยังไม่ได้ตรวจ) 1669

ตอนนี้ต้องการความช่วยเหลือเรื่องใด หรือเพียงแจ้งให้ทราบ กรุณาบอกได้เลยค่ะ
👤 ต้องตัดไฟตอนไหนคะ
   ✦ intents=['ask_information'] target=active_incident
   ➜ actions=['ask_follow_up', 'answer_question']
